In [3]:
from langchain_classic.vectorstores.cassandra import Cassandra
from langchain_classic.indexes.vectorstore import VectorStoreIndexWrapper
from langchain_community.llms import Ollama
from langchain_community.embeddings import OllamaEmbeddings

# Support for dataset retrieval with Hugging Face
from datasets import load_dataset
from PyPDF2 import PdfReader


c:\Users\003X65744\Desktop\Data Science\Chatbot\chatenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
pdfreader = PdfReader('pdfs/confident.pdf')

In [6]:
from typing_extensions import Concatenate
# read text from pdf
raw_text = ''
for i, page in enumerate(pdfreader.pages):
    content = page.extract_text()
    if content:
        raw_text += content

In [7]:
raw_text

'Digital \nArticle\nPersonal Strategy and \nStyle\n6 Ways to Look More \nConfident During a \nPresentation\nHere’s what the best leaders do. by\xa0Kasia\xa0Wezowski\nThis article is licensed to you, Bikash Ojha of International Business Machines (IBM), for your personal use through 2024-01-01. Further posting, copying or distribution is not permitted. \nCopyright 2017-04-06 Harvard Business Publishing. All rights reserved.6 Ways to Look More \nConfident During a \nPresentation\nHere’s what the best leaders do.  by\xa0Kasia \xa0Wezowski\nPublished on HBR.org   /  April 06, 2017   /  Reprint H03ETV\nSeveral years ago, colleagues and I were  invited to predict the results \nof a start-up pitch contest in Vienna, where 2,500 tech entrepreneurs \nwere competing to win thousands of euros in funds. We observed\xa0the \npresentations, but rather than paying attention to the ideas the \nentrepreneurs were pitching, we were watching the body language and \nmicroexpressions  of the judges as they

In [20]:
llm = Ollama(model = "llama3.2:1b")
embedding = OllamaEmbeddings(model="llama3.2:1b")

In [28]:
from langchain_astradb import AstraDBVectorStore
from dotenv import load_dotenv
load_dotenv()
import os

astra_vector_store = AstraDBVectorStore(
    embedding=embedding,
    collection_name="my_chat_tablee1",
    api_endpoint=os.getenv("ASTRA_EP"),
    token=os.getenv("ASTRA_TOKEN")
)

In [29]:
from langchain_classic.text_splitter import CharacterTextSplitter
# We need to split the text using Character Text Split such that it sshould not increse token size
text_splitter = CharacterTextSplitter(
    separator = "\n",
    chunk_size = 800,
    chunk_overlap  = 200,
    length_function = len,
)
texts = text_splitter.split_text(raw_text)

In [30]:
texts[:50]

['Digital \nArticle\nPersonal Strategy and \nStyle\n6 Ways to Look More \nConfident During a \nPresentation\nHere’s what the best leaders do. by\xa0Kasia\xa0Wezowski\nThis article is licensed to you, Bikash Ojha of International Business Machines (IBM), for your personal use through 2024-01-01. Further posting, copying or distribution is not permitted. \nCopyright 2017-04-06 Harvard Business Publishing. All rights reserved.6 Ways to Look More \nConfident During a \nPresentation\nHere’s what the best leaders do.  by\xa0Kasia \xa0Wezowski\nPublished on HBR.org   /  April 06, 2017   /  Reprint H03ETV\nSeveral years ago, colleagues and I were  invited to predict the results \nof a start-up pitch contest in Vienna, where 2,500 tech entrepreneurs \nwere competing to win thousands of euros in funds. We observed\xa0the',
 'of a start-up pitch contest in Vienna, where 2,500 tech entrepreneurs \nwere competing to win thousands of euros in funds. We observed\xa0the \npresentations, but rather tha

In [31]:
astra_vector_store.add_texts(texts[:50])

print("Inserted %i headlines." % len(texts[:50]))

astra_vector_index = VectorStoreIndexWrapper(vectorstore=astra_vector_store)

Inserted 17 headlines.


In [32]:
first_question = True
while True:
    if first_question:
        query_text = input("\nEnter your question (or type 'quit' to exit): ").strip()
    else:
        query_text = input("\nWhat's your next question (or type 'quit' to exit): ").strip()

    if query_text.lower() == "quit":
        break

    if query_text == "":
        continue

    first_question = False

    print("\nQUESTION: \"%s\"" % query_text)
    answer = astra_vector_index.query(query_text, llm=llm).strip()
    print("ANSWER: \"%s\"\n" % answer)

    print("FIRST DOCUMENTS BY RELEVANCE:")
    for doc, score in astra_vector_store.similarity_search_with_score(query_text, k=4):
        print("    [%0.4f] \"%s ...\"" % (score, doc.page_content[:84]))


QUESTION: "what is clinton box ?"
ANSWER: "According to the text, "the Clinton box" refers to a gesture where one imagines a box in front of their chest and belly, containing one's hand movements within it. This gesture is used to signal confidence and control during presentations."

FIRST DOCUMENTS BY RELEVANCE:
    [0.7621] "The box
HBR  /  Digital Article   /  6 Ways to Look More Confident During a Presenta ..."
    [0.7582] "width apart, it signals that you feel in control.
Palms up
HBR  /  Digital Article   ..."
    [0.7499] "analyzed them for six key emotions identified  in psychology research: 
happy, surpr ..."
    [0.7471] "imagine a box in front of his chest and belly and contain his hand 
movements within ..."
